# Baseline check: `_upper` forcing vs calibrated run

Compares the `.his` output of the Baseline re-run with the rebuilt
`*_remapped_upper.bin` forcing against the calibrated run (Run42,
`his_scenario_files`). Self-contained: `load_his` is copied unchanged from
`Torun_WAQ_Analysis_netcdf_20260603`.

**Expected:** `RadAve` identical (233.5 in layer 8, 0 in layers 9–16);
phytoplankton and other variables close but not necessarily identical, because
43 surface cells now receive light and wind.

## 1 — Setup

In [ ]:
import os, struct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

BASE   = r'\\luna\fst\LEC\Users\midauarg\delft_modelling\wq_0.1min_secondfolder'
T0     = datetime(2022, 9, 9, 0, 0, 0)   # reference time, same as the analysis notebook
CUTOFF = '2023-09-01'                    # set to None to compare the full period

OLD_FOLDER, OLD_RUNID = 'his_scenario_files', 'Torun_WAQ_Apr2023'   # calibrated Run42
NEW_FOLDER, NEW_RUNID = '<folder of the new Baseline .his, relative to BASE>', 'Torun_WAQ_Apr2023'  # <-- edit

In [ ]:
def load_his(folder, runid='Torun_WAQ_Apr2023'):
    """Load a DELWAQ binary .his file. Returns dict with data and metadata."""
    his_file = os.path.join(BASE, folder, f'{runid}.his')
    with open(his_file, 'rb') as f:
        raw = f.read()
    pos   = 160  # skip 4×40-char description header
    n_sub = struct.unpack_from('<i', raw, pos)[0]; pos += 4
    n_loc = struct.unpack_from('<i', raw, pos)[0]; pos += 4
    substances = []
    for _ in range(n_sub):
        substances.append(raw[pos:pos+20].decode('latin-1').strip()); pos += 20
    locations = []
    for _ in range(n_loc):
        pos += 4  # skip 4-byte index prefix
        locations.append(raw[pos:pos+20].decode('latin-1').strip()); pos += 20
    rec_size  = 4 + n_loc * n_sub * 4
    n_times   = (len(raw) - pos) // rec_size
    times_sec = np.zeros(n_times, dtype=np.int32)
    data      = np.zeros((n_times, n_loc, n_sub), dtype=np.float32)
    for t in range(n_times):
        times_sec[t] = struct.unpack_from('<i', raw, pos)[0]; pos += 4
        vals = np.frombuffer(raw, dtype='<f4', count=n_loc*n_sub, offset=pos)
        data[t] = vals.reshape(n_loc, n_sub); pos += n_loc * n_sub * 4
    timestamps = [T0 + timedelta(seconds=int(s)) for s in times_sec]
    ts_index   = pd.DatetimeIndex(timestamps)

    def find_sub(name):
        for i, s in enumerate(substances):
            if name.lower() in s.lower(): return i
        return None
    cab_locs, bis_locs = {}, {}
    for i, loc in enumerate(locations):
        l = loc.strip()
        if l.startswith('CAB (') and l.endswith(')'):
            cab_locs[int(l[5:-1].strip())] = i
        elif l.startswith('BIS (') and l.endswith(')'):
            bis_locs[int(l[5:-1].strip())] = i
    idx = {s: i for i, s in enumerate(substances)}
    legacy_names = ['Continuity','OXY','NH4','NO3','PO4','Si',
                    'POC1','PON1','POP1','AAP','Opal',
                    'APHANFIX_E','FDIATOMS_E','FFLAGELA',
                    'GREENS_E','MICROCYS_E','OSCILAT_E','ANABAENA_E',
                    'APHANFIX_N','GREENS_N','MICROCYS_N','OSCILAT_N','ANABAENA_N',
                    'APHANFIX_P','GREENS_P','MICROCYS_P','OSCILAT_P','ANABAENA_P',
                    'FDIATOMS_P','MICROCYS_N','MICROCYS_P',
                    'DetCS1','DetNS1','DetPS1','DetSiS1','AAPS1','SatPercOXY',
                    'Chlfa','SedOXYDem',
                    'Limit Chlo','Limit nit','Limit pho','Limit sil',
                    'Limit e','Limit gro',
                    'RadAve','VWind','Temp','ExtVl']
    for name in legacy_names:
        if name not in idx:
            found = find_sub(name)
            if found is not None:
                idx[name] = found
    print(f"Loaded: {folder}")
    print(f"  {n_sub} substances, {n_loc} locations, {n_times} timesteps")
    print(f"  {timestamps[0].date()} -> {timestamps[-1].date()}")
    print(f"  idx contains {len(idx)} substances - Chlfa: {idx.get('Chlfa')}")
    if CUTOFF:
        mask       = ts_index <= pd.Timestamp(CUTOFF)
        data       = data[mask]
        timestamps = [t for t, m in zip(timestamps, mask) if m]
        ts_index   = pd.DatetimeIndex(timestamps)
    return dict(data=data, timestamps=timestamps, ts_index=ts_index,
                substances=substances, locations=locations,
                cab_locs=cab_locs, bis_locs=bis_locs,
                cab_avg=next((i for i,l in enumerate(locations) if 'CAB_av' in l), None),
                bis_avg=next((i for i,l in enumerate(locations) if 'BIS_av' in l), None),
                idx=idx)

## 2 — Load both runs

In [ ]:
old = load_his(OLD_FOLDER, runid=OLD_RUNID)
new = load_his(NEW_FOLDER, runid=NEW_RUNID)

assert old['substances'] == new['substances'], 'substance lists differ'
assert old['locations']  == new['locations'],  'location lists differ'
assert old['ts_index'].equals(new['ts_index']), 'time axes differ'

def raw(run, sub, loc):
    # Raw .his value (no Continuity division - RadAve/Limit e are process outputs)
    return run['data'][:, loc, run['substances'].index(sub)]

PHYTO = ['FDIATOMS_E','FDIATOMS_P','FFLAGELA','GREENS_E','GREENS_N','GREENS_P',
         'MICROCYS_E','MICROCYS_N','MICROCYS_P']
pd.set_option('display.float_format', '{:.4g}'.format)

## 3 — Per layer: light and phytoplankton

In [ ]:
rows = []
for station, key in [('CAB', 'cab_locs'), ('BIS', 'bis_locs')]:
    for layer, loc in sorted(old[key].items()):
        p_o = sum(raw(old, s, loc) for s in PHYTO)
        p_n = sum(raw(new, s, loc) for s in PHYTO)
        rows.append(dict(station=station, layer=layer,
                         RadAve_old=raw(old, 'RadAve', loc).mean(),
                         RadAve_new=raw(new, 'RadAve', loc).mean(),
                         LimE_old=raw(old, 'Limit e', loc).mean(),
                         LimE_new=raw(new, 'Limit e', loc).mean(),
                         phyto_old=p_o.mean(), phyto_new=p_n.mean(),
                         phyto_diff_pct=100 * (p_n.mean() - p_o.mean()) / p_o.mean()))
layer_df = pd.DataFrame(rows).set_index(['station', 'layer'])
display(layer_df)

## 4 — Every output variable, depth-averaged locations (largest % change first)

In [ ]:
rows = []
for lname in ['CAB_averaged', 'BIS_averaged']:
    loc = old['locations'].index(lname)
    for s in old['substances']:
        a, b = raw(old, s, loc), raw(new, s, loc)
        rows.append(dict(location=lname, substance=s, mean_old=a.mean(), mean_new=b.mean(),
                         diff_pct=100 * (b.mean() - a.mean()) / a.mean() if a.mean() != 0 else np.nan,
                         max_abs_diff=np.abs(b - a).max()))
all_df = pd.DataFrame(rows).set_index(['location', 'substance'])
display(all_df.sort_values('diff_pct', key=abs, ascending=False).head(20))

## 5 — Time series at CAB

In [ ]:
loc_s = old['cab_locs'][8]                       # top active layer
loc_a = old['locations'].index('CAB_averaged')
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for ax, (lab, f) in zip(axes, [
        ('Phyto, layer 8',      lambda r: sum(raw(r, s, loc_s) for s in PHYTO)),
        ('Chlfa, CAB_averaged', lambda r: raw(r, 'Chlfa', loc_a)),
        ('OXY, layer 8',        lambda r: raw(r, 'OXY', loc_s))]):
    ax.plot(old['ts_index'], f(old), lw=1, label='calibrated (Run42)')
    ax.plot(new['ts_index'], f(new), lw=1, ls='--', label='new Baseline (_upper)')
    ax.set_ylabel(lab, fontsize=8)
axes[0].legend(fontsize=7)
plt.tight_layout()